In [ ]:
import tifffile as tf 
from pathlib import Path
import skimage
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import normalized_mutual_information as nmi
import matplotlib.pyplot as plt
import numpy as np
import cv2
import pickle 
from tqdm import tqdm 
import seaborn as sns 
import pandas as pd
import torch
!pip install pytorch-msssim
from pytorch_msssim import ms_ssim

In [ ]:
keys = [ 
        '/Multiplex 2/CTRL 7',
        '/Multiplex 6/MP3-058-2', 
        '/Multiplex 6/L554-5', 
        '/Multiplex 6/CTRL 4', 
        '/Multiplex 8/Med 2',
        '/Multiplex 9/L554-4', 
        '/Multiplex 2/ANCA 9',
        '/Multiplex 11/L554-3'
        ]

In [ ]:
keys_name = {
    '/Multiplex 2/CTRL 7': 'HKT CTRL 1',
    '/Multiplex 6/MP3-058-2': 'HKT PVAN', 
    '/Multiplex 6/L554-5': 'PKT Allo 1', 
    '/Multiplex 6/CTRL 4': 'HKT CTRL 2', 
    '/Multiplex 8/Med 2': 'HKT Infection',
    '/Multiplex 9/L554-4': 'PKT Allo 2', 
    '/Multiplex 2/ANCA 9': 'HKT ANCA'
}

In [ ]:
registration_path = 'Y:/coskun-lab/Aditi/01/Results/registration'

In [ ]:
#Load in registered images and compute SSIM between image pairs 
ssim_score = {}

for key in tqdm(keys):
    
    if(key == '/Multiplex 2/ANCA 9'):
        key_registration_path = registration_path + key + '/backup'
    else:
        key_registration_path = registration_path + key + '/1x'
    dapi_paths = [f for f in Path(key_registration_path).rglob('*.tif') if 'registered' in str(f) and 'CH1' in str(f)]
    color_paths =  [f for f in Path(key_registration_path).rglob('*.tif') if 'registered' in str(f) and 'Cycle' not in str(f)]
    channels = {}

    for channel in dapi_paths:
        channels[str(channel).split('\\')[-1].split('.')[0].split('_')[0]] = tf.imread(channel)

    for channel in color_paths:
        channels[str(channel).split('\\')[-1].split('.')[0].split('_')[0]] = tf.imread(channel)
    ssim_score[keys_name[key]] = []


    i = 0
    for channel_name, image in channels.items():
        print(channel_name)
        if(i == 0):
            i+=1
            continue
        if('Cycle' in channel_name):
            reference = list(channels.values())[0].copy().astype(np.float32)
            reference[reference <= 0.02] = 0
            image_to_register = image.copy().astype(np.float32)
            image_to_register[image_to_register <= 0.02] = 0

            reference = cv2.normalize(reference, None, 0, 1, cv2.NORM_MINMAX)
            image_to_register = cv2.normalize(image_to_register, None, 0, 1, cv2.NORM_MINMAX)
            data_range = max(
                image_to_register.max() - image_to_register.min(),
                reference.max() - reference.min()
            )
            ssim_score[keys_name[key]].append(ssim(reference, image_to_register, data_range=data_range))
        else:
            if('H&E' in channel_name):
                reference = list(channels.values())[0].copy().astype(np.float32)
                reference[reference <= 0.02] = 0
                reference = cv2.normalize(reference, None, 0, 1, cv2.NORM_MINMAX)
            else:
                reference =  skimage.color.rgb2gray(channels['H&E'].copy()).astype(np.float32)
            
            image_gray = skimage.color.rgb2gray(image).astype(np.float32)
            image_edges =  cv2.normalize(cv2.Laplacian(image_gray, cv2.CV_32F), None, 0, 1, cv2.NORM_MINMAX)

            reference_edges = cv2.normalize(cv2.Laplacian(reference, cv2.CV_32F), None, 0, 1, cv2.NORM_MINMAX)

            data_range = max(
                image_edges.max() - image_edges.min(),
                reference_edges.max() - reference_edges.min()
            )

            ssim_score[keys_name[key]].append(ssim(image_edges, reference_edges, data_range=data_range))


In [ ]:
ssim_score

In [ ]:
with open(registration_path + '/registration_ms_ssim.pickle', 'wb') as f: 
    pickle.dump(ssim_score, f)

In [ ]:
import matplotlib as mpl 
mpl.rcParams["font.size"] = 25

ssim_score_all = []
for sample, quality_scores in ssim_score.items():
    for score in quality_scores:
        ssim_score_all.append({'Sample': sample, 'Image Quality': score})  #.split('/')[-1]

ssim_score_all_df = pd.DataFrame(ssim_score_all)
plt.figure(figsize=(12, 6))

ax = sns.boxplot(data=ssim_score_all_df, x='Sample', y='Image Quality', palette='tab20')

ax.set_xticklabels(ax.get_xticklabels(),rotation=45)

sns.stripplot(data=ssim_score_all_df, x='Sample', y='Image Quality', jitter=True, color='black', alpha=0.6)

plt.xlabel("Sample")
plt.ylabel("SSIM")
plt.ylim(0, 1)

plt.show()
